# Stepper motor dynamic response identification

This notebook measures the STEVAL-EDUKIT01 rotor response to small position-step commands and estimates position, velocity, acceleration, and a second-order actuator model similar to the UCLA/ST instructor manual.

The model used in the manual is `G_rotor(s) = a / (s^2 + b s + c)` with unity DC gain implying `a = c` (sign depends on angle convention).

**Safety:** start with the pendulum removed or hanging down, rotor centered, and small step amplitudes. Keep hands clear. The firmware rotor angle is based on internal step count; it cannot directly detect missed steps.

In [ ]:
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, optimize
from control_comms import ControlComms, StatusCode, DebugLevel

## Configuration

The current firmware uses a fixed L6474 profile unless it is recompiled. Record those values here so every dataset contains the test configuration. The repository default is 2000 pps max speed, 30 pps min speed, 6000 pps^2 acceleration/deceleration, 1/16 microstep and 800 mA TVAL.

In [ ]:
SERIAL_PORT = 'COM6'       # change for your computer
BAUD_RATE = 500000
TIMEOUT = 0.25
SAMPLE_PERIOD_S = 0.01     # target host polling period
STEP_AMPLITUDE_DEG = 15.0
PRE_STEP_S = 0.5
HOLD_S = 2.0
POST_STEP_S = 1.0
N_REPEATS = 3

PROFILE = {
    'max_speed_pps': 2000,
    'min_speed_pps': 30,
    'accel_pps2': 6000,
    'decel_pps2': 6000,
    'microstep': 16,
    'tval_mA': 800,
}

CMD_SET_HOME = 0
CMD_MOVE_TO = 1
CMD_HARD_STOP = 5
CMD_QUERY = 6
CMD_RESET_SAFETY = 7

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
ctrl = ControlComms(timeout=TIMEOUT, debug_level=DebugLevel.DEBUG_ERROR)
ret = ctrl.connect(SERIAL_PORT, BAUD_RATE)
if ret is not StatusCode.OK:
    raise RuntimeError(f'Could not connect to {SERIAL_PORT}')

ctrl.step(CMD_HARD_STOP, [0.0])
ctrl.step(CMD_RESET_SAFETY, [0.0])
resp = ctrl.step(CMD_SET_HOME, [0.0])
print('Connected:', resp)

## Acquisition helper

Each query records PC time, MCU time, command target, pendulum angle, rotor angle, L6474 reported step rate, driver status, and safety state. Angular velocity and acceleration are computed afterward from the rotor-angle time series.

In [ ]:
def sample_once(target_deg, repeat_id, phase):
    t_pc = time.perf_counter()
    resp = ctrl.step(CMD_QUERY, [0.0])
    if resp is None:
        return None
    status, timestamp_ms, terminated, obs = resp
    if len(obs) < 4:
        return None
    return {
        'pc_time_s': t_pc,
        'mcu_time_ms': timestamp_ms,
        'repeat': repeat_id,
        'phase': phase,
        'target_deg': float(target_deg),
        'pendulum_deg': float(obs[0]),
        'rotor_deg': float(obs[1]),
        'motor_speed_pps': float(obs[2]),
        'l6474_status_raw': int(obs[3]),
        'firmware_status': int(status),
        'safety_latched': bool(terminated),
    }

def acquire_for(duration_s, target_deg, repeat_id, phase, rows):
    t0 = time.perf_counter()
    next_t = t0
    while time.perf_counter() - t0 < duration_s:
        row = sample_once(target_deg, repeat_id, phase)
        if row is not None:
            rows.append(row)
            if row['safety_latched']:
                raise RuntimeError('Firmware safety latch triggered')
        next_t += SAMPLE_PERIOD_S
        delay = next_t - time.perf_counter()
        if delay > 0:
            time.sleep(delay)

## Run repeated position-step tests

The command sequence alternates `0 -> +A -> 0 -> -A -> 0`. This keeps the rotor near home and provides both polarities for checking asymmetry.

In [ ]:
rows = []
try:
    for rep in range(N_REPEATS):
        for target in (0.0, STEP_AMPLITUDE_DEG, 0.0, -STEP_AMPLITUDE_DEG, 0.0):
            phase = f'target_{target:+.1f}'
            resp = ctrl.step(CMD_MOVE_TO, [float(target)])
            if resp is None:
                raise RuntimeError('No response to MOVE_TO')
            duration = HOLD_S if target != 0 else PRE_STEP_S
            acquire_for(duration, target, rep, phase, rows)
finally:
    ctrl.step(CMD_HARD_STOP, [0.0])

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError('No data collected')
df['time_s'] = (df['mcu_time_ms'] - df['mcu_time_ms'].iloc[0]) / 1000.0
print(df.shape)
df.head()

## Estimate velocity and acceleration

A Savitzky-Golay filter is applied before differentiation to reduce numerical noise. The L6474 speed value is also converted from pulses/s to deg/s for comparison.

In [ ]:
t = df['time_s'].to_numpy()
phi = df['rotor_deg'].to_numpy()
dt_med = np.median(np.diff(t))
window = max(5, int(round(0.08 / dt_med)) | 1)
window = min(window, len(df) - (1 - len(df) % 2))
if window >= 5:
    phi_smooth = signal.savgol_filter(phi, window_length=window, polyorder=2)
else:
    phi_smooth = phi.copy()
vel_deg_s = np.gradient(phi_smooth, t)
acc_deg_s2 = np.gradient(vel_deg_s, t)
pps_to_deg_s = 360.0 / (200.0 * PROFILE['microstep'])
df['rotor_deg_s_est'] = vel_deg_s
df['rotor_deg_s2_est'] = acc_deg_s2
df['motor_deg_s_from_pps'] = df['motor_speed_pps'] * pps_to_deg_s
df[['time_s','target_deg','rotor_deg','rotor_deg_s_est','rotor_deg_s2_est']].head()

## Fit the manual's second-order rotor model

We fit `G(s) = k*wn^2 / (s^2 + 2*zeta*wn*s + wn^2)`. For unity DC gain `k` should be close to 1. The equivalent manual coefficients are `a=k*wn^2`, `b=2*zeta*wn`, `c=wn^2`. This is a small-signal equivalent model, not an internal motor PID.

In [ ]:
u = df['target_deg'].to_numpy()
y = df['rotor_deg'].to_numpy()
t_fit = t - t[0]

def simulate_second_order(params):
    log_wn, log_zeta, log_k = params
    wn, zeta, k = np.exp(log_wn), np.exp(log_zeta), np.exp(log_k)
    sys = signal.TransferFunction([k * wn**2], [1.0, 2*zeta*wn, wn**2])
    _, yhat, _ = signal.lsim(sys, U=u, T=t_fit)
    return yhat

def residual(params):
    return simulate_second_order(params) - y

x0 = np.log([5.0, 0.7, 1.0])
fit = optimize.least_squares(residual, x0, max_nfev=300)
wn, zeta, k = np.exp(fit.x)
a = k * wn**2
b = 2 * zeta * wn
c = wn**2
yhat = simulate_second_order(fit.x)
rmse = np.sqrt(np.mean((yhat - y)**2))
corr = np.corrcoef(y, yhat)[0,1]
print(f'wn   = {wn:.4f} rad/s')
print(f'zeta = {zeta:.4f}')
print(f'k    = {k:.4f}')
print(f'a={a:.5f}, b={b:.5f}, c={c:.5f}')
print(f'RMSE={rmse:.4f} deg, corr={corr:.5f}')

## Plot measured dynamic response

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
ax[0].plot(t, df['target_deg'], label='command phi_RC')
ax[0].plot(t, df['rotor_deg'], label='measured/internal phi')
ax[0].plot(t, yhat, '--', label='2nd-order fit')
ax[0].set_ylabel('angle [deg]'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(t, df['rotor_deg_s_est'], label='d(phi)/dt')
ax[1].plot(t, df['motor_deg_s_from_pps'], '--', label='L6474 speed converted')
ax[1].set_ylabel('velocity [deg/s]'); ax[1].legend(); ax[1].grid(True)
ax[2].plot(t, df['rotor_deg_s2_est'])
ax[2].set_ylabel('acceleration [deg/s^2]'); ax[2].grid(True)
ax[3].plot(t, df['pendulum_deg'])
ax[3].set_ylabel('pendulum [deg]'); ax[3].set_xlabel('time [s]'); ax[3].grid(True)
fig.suptitle('Stepper / rotor dynamic response')
fig.tight_layout()

## Frequency response of identified model

This reproduces the type of Bode comparison used in the instructor manual. Repeat the experiment after changing the firmware motor profile to compare configurations.

In [ ]:
sys_fit = signal.TransferFunction([a], [1.0, b, c])
w, mag, phase = signal.bode(sys_fit, w=np.logspace(-2, 2, 500))
fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax[0].semilogx(w/(2*np.pi), mag); ax[0].set_ylabel('magnitude [dB]'); ax[0].grid(True, which='both')
ax[1].semilogx(w/(2*np.pi), phase); ax[1].set_ylabel('phase [deg]'); ax[1].set_xlabel('frequency [Hz]'); ax[1].grid(True, which='both')
fig.suptitle('Identified G_rotor frequency response')
fig.tight_layout()

## Save dataset and model parameters

In [ ]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = DATA_DIR / f'stepper_dynamic_response_{stamp}.csv'
model_path = DATA_DIR / f'stepper_dynamic_response_{stamp}_model.csv'
df.to_csv(csv_path, index=False)
pd.DataFrame([{**PROFILE, 'step_amplitude_deg': STEP_AMPLITUDE_DEG, 'wn_rad_s': wn, 'zeta': zeta, 'k': k, 'a': a, 'b': b, 'c': c, 'rmse_deg': rmse, 'correlation': corr}]).to_csv(model_path, index=False)
print(csv_path)
print(model_path)

## Compare with UCLA/ST manual configurations

The manual reports example configurations (all with acceleration/deceleration 3000 step/s^2): high speed Max=1000, Min=300; medium Max=1000, Min=200; low Max=200, Min=200. It reports fitted `(a,b,c)` values approximately `(0.22,0.90,0.44)`, `(0.245,1.12,0.49)`, and `(0.275,1.89,0.55)` respectively.

Do not expect identical coefficients from this notebook because the current Arduino firmware, sample timing, motor profile, mechanical loading, and rotor-position measurement path differ from the original firmware. Use the same identification procedure to build coefficients for your actual configuration.